In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import accuracy_score

# 성별 구별을 위한 데이터 생성
np.random.seed(42)

# 1000명의 데이터를 생성: 나이, 키, 체중 특성
# 나이: 18~60, 키: 150~190 cm, 체중: 40~100 kg
X = np.random.rand(1000, 3).astype(np.float32)
X[:, 0] = X[:, 0] * 43 + 18  # 나이: 18~60
X[:, 1] = X[:, 1] * 40 + 150  # 키: 150~190
X[:, 2] = X[:, 2] * 60 + 40  # 체중: 40~100

# 성별 (0: 여자, 1: 남자) 라벨 생성
y = (X[:, 0] > 30).astype(np.float32)  # 30세 이상이면 남자, 아니면 여자

# 데이터셋 나누기 (훈련과 테스트)
train_size = int(0.8 * len(X))  # 80% 훈련 데이터
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# DataLoader 설정
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 모델 정의
class GenderNN(nn.Module):
    def __init__(self):
        super(GenderNN, self).__init__()
        self.fc1 = nn.Linear(3, 64)  # 3개의 입력 특성 (나이, 키, 체중)
        self.dropout = nn.Dropout(0.5)  # 드롭아웃 적용
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)  # 이진 분류 (여자: 0, 남자: 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # 첫 번째 은닉층
        x = self.dropout(x)  # 드롭아웃 적용
        x = torch.relu(self.fc2(x))  # 두 번째 은닉층
        x = torch.sigmoid(self.fc3(x))  # 출력층
        return x

# 모델 인스턴스
model = GenderNN()

# 손실 함수와 옵티마이저 설정
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)

# 훈련 함수
def train(model, train_loader, criterion, optimizer, epochs=10):
    model.train()  # 훈련 모드로 설정
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # 기울기 초기화

            # 순전파
            outputs = model(inputs)

            # 손실 계산
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()  # 역전파
            optimizer.step()  # 가중치 업데이트

            running_loss += loss.item()

        # 에포크마다 손실 출력
        print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}')

# 훈련 시작
train(model, train_loader, criterion, optimizer, epochs=10)

# 테스트 함수
def test(model, test_loader):
    model.eval()  # 평가 모드로 설정
    all_preds = []
    all_labels = []
    with torch.no_grad():  # 기울기 계산을 하지 않음
        for inputs, labels in test_loader:
            outputs = model(inputs)
            preds = (outputs.squeeze() > 0.5).float() # 0.5를 기준으로 이진 예측
            all_preds.append(preds)
            all_labels.append(labels)

    # 결과를 텐서에서 numpy로 변환
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # 정확도 계산
    accuracy = accuracy_score(all_labels, all_preds)
    print(f'Accuracy on test data: {accuracy * 100:.2f}%')

# 테스트 시작
test(model, test_loader)

Epoch 1/10, Loss: 9.212307306436392
Epoch 2/10, Loss: 2.6251468933545627
Epoch 3/10, Loss: 1.1894036485598638
Epoch 4/10, Loss: 0.9098638938023493
Epoch 5/10, Loss: 0.7271338632473578
Epoch 6/10, Loss: 0.7820069262614617
Epoch 7/10, Loss: 0.612666666507721
Epoch 8/10, Loss: 0.547085668031986
Epoch 9/10, Loss: 0.582526988708056
Epoch 10/10, Loss: 0.5228897516544049
Accuracy on test data: 90.00%
